# 🚀 生成式AI應用開發｜第03週
## Claude API 入門：從 API Key 到第一個 Python AI 函式

> **執行環境：Google Colab** ｜ **本版本使用 Claude API**

本教材內容對應 OpenAI API 入門版，但改用 Anthropic Claude Messages API。重點是比較不同 LLM 平台在 Python SDK、API key、messages、system prompt、回應物件與 <font color="#1a73e8"><b>usage</b></font> 欄位上的差異。

官方文件入口：
- Claude Messages API: https://platform.claude.com/docs/en/build-with-claude/working-with-messages
- Claude model overview: https://platform.claude.com/docs/en/about-claude/models/overview

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>本週任務</b></font><br>
  完成第一個 API 呼叫，並把呼叫流程封裝成可重複使用的 Python 函式。
</div>


> **學生版**：本檔保留課堂練習的 TODO 骨架。執行 Claude API cells 可能產生成本或用量。

## 🎯 本週學習目標

| # | 能力 | Claude 對應概念 |
|---|------|----------------|
| 1 | 安裝與匯入 Claude SDK | `anthropic` |
| 2 | 使用 Colab Secrets 讀取 API key | `ANTHROPIC_API_KEY` |
| 3 | 建立 Claude client | `anthropic.Anthropic()` |
| 4 | 呼叫 Messages API | `client.messages.create()` |
| 5 | 使用 top-level `system` 與 `messages` | 角色設定與對話輸入 |
| 6 | 解析 `message.content[0].text` 與 <font color="#1a73e8"><b>usage</b></font> | 顯示回答、除錯、估算成本 |
| 7 | 封裝函式與錯誤處理 | 後續 App 開發 |


---
## 🧰 第一節：安裝套件

In [ ]:
!pip install anthropic --quiet
print("Anthropic SDK 安裝完成")

In [ ]:
import os
import json
import anthropic

print("模組載入完成")

---
## 🔐 第二節：設定 API Key

請在 Claude Console 建立 API key，並在 Colab Secrets 新增：

- Name: `ANTHROPIC_API_KEY`
- Value: 你的 Claude API key
- 開啟 Notebook access

<font color="#b00020"><b>不要把真正的 API key</b></font> 寫在 notebook 裡。

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>安全提醒</b></font><br>
  <font color="#b00020"><b>API key 是機密資訊</b></font>。請使用 Colab Secrets 或環境變數，不要直接寫在 notebook、作業或截圖中。
</div>


In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key
        print(f"已讀取 Claude API key（前 8 碼）：{api_key[:8]}...")
    else:
        print("Secrets 中找不到 ANTHROPIC_API_KEY，請先完成設定")
except Exception as e:
    print(f"目前可能不是在 Colab 執行：{e}")
    print("若在本機執行，請先設定環境變數 ANTHROPIC_API_KEY")

print("目前讀取狀態：", "已設定" if os.getenv("ANTHROPIC_API_KEY") else "未設定")

### 💻 本機環境變數設定方式

```powershell
$env:ANTHROPIC_API_KEY="你的 Claude API key"
setx ANTHROPIC_API_KEY "你的 Claude API key"
```

```bash
export ANTHROPIC_API_KEY="你的 Claude API key"
```

---
## 🔌 第三節：建立 Claude Client

In [ ]:
client = anthropic.Anthropic()

# 課堂預設使用速度快、成本較低的 Haiku。若老師課前確認其他模型更適合，可用 CLAUDE_MODEL 覆蓋。
MODEL = os.getenv("CLAUDE_MODEL", "claude-haiku-4-5")

print(f"Client 建立完成，使用模型：{MODEL}")

---
## ✨ 第四節：第一個 Claude API 呼叫

Claude Messages API 使用 `client.messages.create()`。必填欄位包含 `model`、`max_tokens`、`messages`。

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[
        {"role": "user", "content": "請用繁體中文，用三句話解釋什麼是生成式 AI。"}
    ],
)

print(message.content[0].text)

### 🔍 4-1 檢查 message 物件

Claude 回應中的文字通常在 `message.content[0].text`，<font color="#1a73e8"><b>usage</b></font> 在 `message.<font color="#1a73e8"><b>usage</b></font>`。

In [ ]:
print("message id:", message.id)
print("model:", message.model)
print("text:")
print(message.content[0].text)

print()
print("usage:")
print(message.usage)

---
## 🧭 第五節：加入 system prompt

Claude 會把一開始就適用的角色規則放在 top-level `system` 欄位，而不是把 system 當成第一則 message。

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system="你是一位資管系 Python 助教。回答要精簡、具體，並使用繁體中文。",
    messages=[
        {"role": "user", "content": "請解釋 API key 為什麼不能直接寫在程式碼裡。"}
    ],
)

print(message.content[0].text)

---
## 💬 第六節：用 messages 格式建立多輪輸入

Claude Messages API 是 stateless：每次 API 呼叫都要送出完整對話歷史。這和 Gemini 的 `previous_interaction_id` 不同。

| 平台 | 多輪對話常見做法 |
|---|---|
| OpenAI Responses API | 可用 role/content list 或 previous response |
| Gemini Interactions API | 使用 `previous_interaction_id` |
| Claude Messages API | 每次送出完整 messages history |


In [ ]:
messages = [
    {"role": "user", "content": "我正在學習 Python API 開發。"},
    {"role": "assistant", "content": "很好，API 開發會用到函式、JSON、錯誤處理與環境變數。"},
    {"role": "user", "content": "根據上一句，請建議我下一步該練什麼。"},
]

message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system="你是一位教學助理，回答要適合初學 Python 的大學生。",
    messages=messages,
)

print(message.content[0].text)

### 🧠 6-1 Claude 的 system 與 messages 差異

1. `system`：放整段對話都要遵守的規則。
2. `messages`：放使用者與 assistant 的對話歷史。
3. 多輪對話時，要把先前 user / assistant 訊息一起送出。
4. 做聊天 App 時，需要自己保存 messages list。

---
## 🧩 第七節：封裝成函式

In [ ]:
def ask_ai(user_input, role="你是一位有幫助的助理", model=MODEL):
    # TODO 1：呼叫 client.messages.create()
    # TODO 2：傳入 model、max_tokens、system、messages
    # TODO 3：回傳 message.content[0].text
    return "尚未完成 ask_ai()" 

---
## 🛡️ 第八節：基本錯誤處理

In [ ]:
def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    try:
        # TODO 1：呼叫 Claude API
        # TODO 2：成功時回傳 (True, message.content[0].text)
        return False, "尚未完成 ask_ai_safe()"
    except Exception as e:
        return False, str(e)


success, result = ask_ai_safe("請用一句話說明 JSON 是什麼。")
print(result)

---
## 📊 第九節：讀取 <font color="#1a73e8"><b>usage</b></font> 並建立除錯資訊

In [ ]:
def ask_ai_with_metadata(user_input, role="你是一位有幫助的助理", model=MODEL):
    # TODO 1：呼叫 API
    # TODO 2：回傳 dict，包含 id、model、answer、usage
    return {"id": None, "model": model, "answer": "尚未完成", "usage": None}


result = ask_ai_with_metadata("請列出學習 Claude API 前需要會的 3 個 Python 技能。")
print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

---
## 🧪 第十節：課堂練習

### 📝 練習 A：Python 作業批改助教

請讓 Claude 扮演 Python 作業批改助教，檢查一段縮排錯誤的程式，並輸出「錯誤原因、修正方式、修正版程式碼」。

<div style="border-left:4px solid #188038; padding:10px 12px; background:#e6f4ea; margin:10px 0;">
  <font color="#188038"><b>課堂練習</b></font><br>
  先完成練習 A，再依時間進行練習 B 與選做練習 C。
</div>


In [ ]:
student_code = '''
def add_numbers(a, b):
return a + b

print(add_numbers(3, 5))
'''

role = ""
prompt = ""
success, result = False, "尚未完成課堂練習"
print(result)

---
### 💰 練習 B：<font color="#1a73e8"><b>usage</b></font> 與費用估算

請讀取 `<font color="#1a73e8"><b>usage</b></font>.input_tokens` 與 `<font color="#1a73e8"><b>usage</b></font>.output_tokens`。價格會變動，正式估價前請查官方 pricing。

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>成本提醒</b></font><br>
  本節重點是學會讀取 <font color="#1a73e8"><b>usage</b></font> 與建立估算流程；實際價格請以上課當天官方 pricing 為準。
</div>


In [ ]:
def summarize_usage(usage):
    # TODO：整理 input_tokens 與 output_tokens
    return {"status": "尚未完成"}


result = ask_ai_with_metadata("請用兩句話說明為什麼 API 呼叫需要注意成本。")
print(result["answer"])
print(summarize_usage(result["usage"]))

---
### 🎨 練習 C（選做）：自由設計一個角色

自訂至少兩個 `system`，觀察同一個問題在不同角色設定下的回答差異。

In [ ]:
question = "我想做一個生成式 AI 期末專題，可以從哪裡開始？"
roles = ["", ""]

for i, role in enumerate(roles, 1):
    print(f"===== 角色 {i} =====")
    print("尚未完成")
    print()

---
## ✅ 本週小結

| 技能 | Claude 寫法 |
|------|-------------|
| API key | `ANTHROPIC_API_KEY` |
| Client | `client = anthropic.Anthropic()` |
| 文字生成 | `client.messages.create()` |
| 角色設定 | top-level `system` |
| 使用者訊息 | `messages=[{"role": "user", ...}]` |
| 文字輸出 | `message.content[0].text` |
| 多輪對話 | 每次送出完整 messages history |
